# 03 Exploratory Data Analysis

Use this notebook to explore trends, distributions, segments, anomalies, and early business signals in the Customer Spending Analysis dataset.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()

## 1. Load the Data
Loading the processed master consolidated dataset.

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data/processed/master_consolidated_data.csv'
df = pd.read_csv(DATA_PATH)

# Convert date columns to datetime objects with proper formatting and handling for mixed date strings
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'], format='mixed', dayfirst=True)
df['OpeningDate'] = pd.to_datetime(df['OpeningDate'], format='mixed', dayfirst=True)
df['DateOfBirth'] = pd.to_datetime(df['DateOfBirth'], format='mixed', dayfirst=True)

df.head()

## 2. Dataset Overview
Checking shape, info, missing values, and descriptive statistics.

In [ ]:
print("Dataset Shape:", df.shape)
print("\nData Types and Missing Values:")
print(df.info())
print("\nMissing Value Count:\n", df.isnull().sum())

In [ ]:
df.describe(include='all', datetime_is_numeric=True).T

## 3. Univariate Analysis
Exploring individual distributions such as Transaction Amount, Account Balance, and Transaction Types.

In [ ]:
# Distribution of Transaction Amounts
plt.figure(figsize=(12, 5))
sns.histplot(df['Amount'], bins=50, kde=True, color='blue')
plt.title('Distribution of Transaction Amounts')
plt.xlabel('Transaction Amount')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Distribution of Account Balances
# Since balances repeat per transaction, we group by AccountID to get unique account balances
unique_accounts = df.drop_duplicates(subset=['AccountID'])

plt.figure(figsize=(12, 5))
sns.histplot(unique_accounts['Balance'], bins=50, kde=True, color='green')
plt.title('Distribution of Account Balances')
plt.xlabel('Account Balance')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Count of Transaction Types
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='TransactionTypeID', palette='viridis')
plt.title('Count of Transactions by Type')
plt.xlabel('Transaction Type ID')
plt.ylabel('Count')
plt.show()

## 4. Bivariate Analysis
Investigating relationships between variables (e.g., Transaction Amount over time, or by Customer Type).

In [ ]:
# Transaction Amounts by Transaction Type
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='TransactionTypeID', y='Amount', palette='Set2')
plt.title('Transaction Amounts by Transaction Type')
plt.xlabel('Transaction Type ID')
plt.ylabel('Transaction Amount')
plt.show()

In [ ]:
# Transaction Volume Over Time (Monthly)
df_time = df.dropna(subset=['TransactionDate']).copy()
df_time['Month_Year'] = df_time['TransactionDate'].dt.to_period('M')
monthly_counts = df_time.groupby('Month_Year').size().reset_index(name='TransactionCount')
monthly_counts['Month_Year'] = monthly_counts['Month_Year'].astype(str)

plt.figure(figsize=(14, 6))
sns.lineplot(data=monthly_counts, x='Month_Year', y='TransactionCount', marker='o', color='purple')
plt.title('Monthly Transaction Volume')
plt.xlabel('Month-Year')
plt.ylabel('Number of Transactions')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Outlier Detection
Identifying anomalies in transaction patterns.

In [ ]:
# Scatter plot of Balance vs Transaction Amount
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Balance', y='Amount', alpha=0.5, color='darkorange')
plt.title('Account Balance vs Transaction Amount')
plt.xlabel('Account Balance')
plt.ylabel('Transaction Amount')
plt.show()

## 6. Key Findings / Business Signals
- Record any findings about typical user behavior, unexpected spikes in transaction amounts, or specific transaction types that dominate.